# Aegis.dv – Exploratory Clinical & Operational Data Analysis

**Project:** Aegis.dv – Hospital Operations & Patient Analytics Dashboard  
**Author:** Ananya Biju (Infosys Springboard Internship)  
**Dataset:** Primary Clinical Encounter Data (101,766 Encounters)  

This notebook conducts in-depth exploratory data analysis across patient flow patterns, length-of-stay dynamics, 30-day readmission risk factors, and department workloads.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

df = pd.read_csv(project_root / 'data' / 'processed' / 'hospital_cleaned.csv', low_memory=False)
print(f"Loaded analytical dataset: {len(df):,} encounters")

## 1. Length of Stay (LOS) Distribution & Skewness Analysis

In [ ]:
los = df['time_in_hospital']
print('Length of Stay Summary (Days):')
print(f"  Mean:   {los.mean():.2f} days")
print(f"  Median: {los.median():.2f} days")
print(f"  Std:    {los.std():.2f} days")
print(f"  P75:    {los.quantile(0.75):.2f} days")
print(f"  P90:    {los.quantile(0.90):.2f} days")
print(f"  P99:    {los.quantile(0.99):.2f} days")

print('\nLOS Stay Duration Breakdown:')
print(df['los_band'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 2. 30-Day Readmission Analysis by Clinical & Demographic Risk Factors

In [ ]:
eligible_df = df[df['is_readmission_eligible'] == 1]
overall_rate = eligible_df['readmitted_30d_flag'].mean() * 100
print(f"Overall 30-Day Readmission Rate: {overall_rate:.2f}%")

print('\nReadmission Rate by Age Bracket:')
age_readmit = eligible_df.groupby('age_bracket')['readmitted_30d_flag'].agg(['count', 'mean'])
age_readmit['rate_pct'] = (age_readmit['mean'] * 100).round(2)
print(age_readmit[['count', 'rate_pct']])

print('\nReadmission Rate by Admission Type:')
adm_readmit = eligible_df.groupby('admission_type_group')['readmitted_30d_flag'].agg(['count', 'mean'])
adm_readmit['rate_pct'] = (adm_readmit['mean'] * 100).round(2)
print(adm_readmit[['count', 'rate_pct']])

## 3. Department Volume & Workload Proxy (Patient-Days)

In [ ]:
dept_perf = eligible_df.groupby('department').agg(
    admissions=('encounter_id', 'count'),
    patient_days=('time_in_hospital', 'sum'),
    alos=('time_in_hospital', 'mean'),
    readmissions=('readmitted_30d_flag', 'sum'),
)
dept_perf['readmit_rate_pct'] = (dept_perf['readmissions'] / dept_perf['admissions'] * 100).round(2)
dept_perf['alos'] = dept_perf['alos'].round(2)
dept_perf = dept_perf.sort_values(by='admissions', ascending=False)
print(dept_perf)